# Session 2: Multiple Linear Regression with scikit-learn

## Overview

Session 1 forecast the power plant's output from the temperature alone, with a straight line fitted by hand. Here
you use **all four weather inputs** and let scikit-learn do the fitting. The algorithm is the same least-squares
line, now in four dimensions; the new skills are around it: reading coefficients correctly, choosing which inputs
to keep, adding curvature with polynomial features, and estimating performance honestly with cross-validation.

By the end, you will be able to:

- Fit and evaluate a multiple linear regression with scikit-learn's `fit` / `predict` API.
- Interpret coefficients "holding the other inputs constant", and compare them using standardised inputs.
- Explain why correlated inputs change each other's coefficients.
- Compare feature subsets and model settings with k-fold cross-validation, without touching the test set.
- Add polynomial features inside a `Pipeline` and recognise when extra flexibility stops helping.

**Prerequisites:** Session 1.

**Dataset:** [UCI Combined Cycle Power Plant](https://archive.ics.uci.edu/dataset/294/combined+cycle+power+plant),
as in Session 1: 9,568 hourly averages of ambient temperature, exhaust vacuum, ambient pressure, relative humidity,
and net electrical output (MW).

## Where this fits

| Part | Sessions | Task | Algorithms |
| --- | --- | --- | --- |
| 1. Regression | 1, 2, 3, 4 | Predict a number | Linear regression by gradient descent and in scikit-learn, tree ensembles, explaining black-box models **← you are here** |
| 2. Classification | 5, 6, 7, 8, 9 | Predict a category | Logistic regression, naive Bayes, linear text classifiers, comparing classifiers |
| 3. Clustering | 10, 11 | Find groups without labels | k-means, hierarchical clustering |
| 4. Topic modelling | 12 | Find themes in text | LDA, NMF |

**This session's question:** How much better is the forecast with all four weather inputs, and which of them matter?

## Step 1: Set up the environment

This module has one pinned `requirements.txt` (in the module folder) for all twelve sessions. If you have
not done it yet, create a virtual environment and install it once, from the module folder:

```bash
python3.11 -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\Activate.ps1
pip install -r requirements.txt
jupyter notebook
```

This session uses:

| Package | Why we need it |
| --- | --- |
| ucimlrepo | Downloads the dataset from UCI |
| scikit-learn | Models, pipelines, cross-validation, metrics |
| pandas | Tables |
| numpy | Arrays |
| matplotlib | Charts |

Run the cell below. It imports what this notebook needs and prints the versions. The numbers quoted in
this notebook come from these exact versions; a different version can shift the last decimal place.

In [ ]:
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from importlib.metadata import version

for package in ["ucimlrepo", "scikit-learn", "pandas", "numpy", "matplotlib"]:
    print(f"{package:13s} {version(package)}")

## Step 2: Load the data with the same split as Session 1

The same `random_state=42` gives exactly the same 1,914 test hours as Session 1, so the results are directly
comparable. In scikit-learn the inputs are a table `X` (one row per hour, one column per input) and the target is
a vector `y`.

In [ ]:
from ucimlrepo import fetch_ucirepo

plant = fetch_ucirepo(id=294).data.original.rename(columns={
    "AT": "temperature", "V": "vacuum", "AP": "pressure", "RH": "humidity", "PE": "output"})
FEATURES = ["temperature", "vacuum", "pressure", "humidity"]
train, test = train_test_split(plant, test_size=0.2, random_state=42)
X_train, y_train = train[FEATURES], train["output"]
X_test, y_test = test[FEATURES], test["output"]
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

## Step 3: How the inputs relate to the output, and to each other

Correlations with the output show which inputs carry signal. Correlations *between* inputs warn us that some of
them carry the same signal twice.

In [ ]:
print(train.corr().round(2))

**Read the output:** the bottom row (or right column) shows each input's correlation with the output:
temperature -0.95, vacuum -0.87, pressure +0.52, humidity +0.39. But the inputs are also correlated with each
other: **temperature and vacuum at +0.84**. Hot days come with a high exhaust vacuum, so much of what vacuum
"knows" about output, temperature already knows. Step 5 shows what this does to the coefficients.

## Step 4: Fit a multiple linear regression

The model is now

$$\hat y = b + m_1\,\text{temperature} + m_2\,\text{vacuum} + m_3\,\text{pressure} + m_4\,\text{humidity}$$

Every scikit-learn model works the same way: create it, `fit(X, y)` on the training data, then `predict(X)`.

In [ ]:
def report(name, y_true, y_pred):
    print(f"{name:28s} MAE = {mean_absolute_error(y_true, y_pred):5.2f} MW   "
          f"RMSE = {mean_squared_error(y_true, y_pred) ** 0.5:5.2f} MW   R² = {r2_score(y_true, y_pred):.3f}")


model = LinearRegression().fit(X_train, y_train)
print(f"intercept: {model.intercept_:.2f}")
print(pd.Series(model.coef_, index=FEATURES, name="coefficient").round(4))
print()
one_input = LinearRegression().fit(X_train[["temperature"]], y_train)
report("temperature only (Session 1)", y_test, one_input.predict(X_test[["temperature"]]))
report("all four inputs", y_test, model.predict(X_test))

**Read the output:** the four-input model cuts the test RMSE from 5.38 MW (Session 1) to 4.50 MW, and R² rises
from 0.900 to 0.930.

Each coefficient is the change in output for one unit more of that input, **holding the other three constant**:

- **temperature -1.99:** one degree warmer, same vacuum, pressure and humidity: about 2 MW less.
- **vacuum -0.23:** one cm Hg more exhaust vacuum: 0.23 MW less.
- **pressure +0.06** per millibar and **humidity -0.16** per percentage point.

The test set has now been used once, to report this result. From here on, every choice (which inputs, how much
curvature) is made with cross-validation on the training set only, and the test set is used once more at the
very end.

## Step 5: Coefficients depend on the other inputs

Compare each input's coefficient when it is the only input with its coefficient in the full model.

In [ ]:
alone = {f: LinearRegression().fit(X_train[[f]], y_train).coef_[0] for f in FEATURES}
print(pd.DataFrame({"alone": alone, "in full model": dict(zip(FEATURES, model.coef_))}).round(3))

**Read the output:** temperature barely changes (-2.18 alone, -1.99 together), but **vacuum shrinks from -1.17 to
-0.23** and **pressure from +1.49 to +0.06**: alone, each takes credit for the effect of the temperature it is
correlated with. **Humidity even flips sign**, from +0.46 alone to -0.16 in the full model: humid hours tend to be
cool hours, so alone humidity looks helpful, but at a fixed temperature more humidity slightly lowers output.

Lesson: **a coefficient is never "the effect" of an input on its own**; it is its effect *given the other inputs
in the model*. Add or remove an input and the others can change, even flip.

## Step 6: Which input matters most? Standardise first

The raw coefficients cannot be compared with each other: they are in different units (°C, cm Hg, millibar, %),
and pressure varies far less than vacuum. Standardising every input (Session 1, Step 7) puts the coefficients on
a common scale: **MW per standard deviation** of each input.

`make_pipeline` chains the scaler and the model into one object. Fitting the pipeline fits the scaler on the
training data, and predicting reuses the training mean and standard deviation automatically. This makes it
impossible to scale the test set with its own statistics by mistake.

In [ ]:
scaled = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train, y_train)
coefs = pd.Series(scaled[-1].coef_, index=FEATURES).sort_values(key=abs, ascending=False)
print("MW per standard deviation of each input:")
print(coefs.round(2))

**Read the output:** temperature dominates at -14.8 MW per standard deviation, followed by vacuum (-3.0) and
humidity (-2.3); pressure is smallest (+0.4). The predictions are identical to Step 4: standardising changes the
units of the coefficients, not the model.

## Step 7: Choose the inputs with cross-validation

Do we need all four inputs? Comparing subsets on the test set would turn it into part of the training process,
and its score would no longer be honest. Instead, **k-fold cross-validation** splits the *training* set into k
parts (folds), fits on k-1 and scores on the remaining one, k times, then averages.

With four inputs there are only 15 non-empty subsets, so we can try them all.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=0)


def cv_rmse(model, columns):
    scores = cross_val_score(model, X_train[list(columns)], y_train, cv=cv,
                             scoring="neg_root_mean_squared_error")
    return -scores.mean()


results = {", ".join(cols): cv_rmse(LinearRegression(), cols)
           for k in range(1, 5) for cols in combinations(FEATURES, k)}
print(pd.Series(results, name="CV RMSE (MW)").sort_values().round(3).to_string())

**Read the output:** the full four-input model is best, with a cross-validated RMSE of 4.574 MW. Dropping
pressure costs almost nothing (4.584); dropping humidity costs more (4.903). Temperature alone gives 5.438, close
to Session 1's test RMSE. Every subset without temperature is far worse (7.6 MW and above).

scikit-learn's scorers follow the rule "higher is better", so error metrics come back negated
(`neg_root_mean_squared_error`); we flip the sign back.

We keep all four inputs: pressure adds little, but it costs nothing to measure and does not hurt.

## Step 8: Add curvature with polynomial features

Session 1's residuals showed a bend that a straight line cannot follow. A linear model can still fit curves if we
give it **new inputs**: squares and products of the originals. `PolynomialFeatures(degree=2)` turns the 4 inputs
into 14: the originals, their squares, and every pair multiplied together (for example temperature × vacuum).
The model is still *linear in its coefficients*, so it is fitted in exactly the same way.

The higher the degree, the more flexible the model. Compare training and cross-validated error by degree.

In [ ]:
rows = []
for degree in range(1, 9):
    pipe = make_pipeline(PolynomialFeatures(degree, include_bias=False), StandardScaler(), LinearRegression())
    n_features = pipe[0].fit(X_train).n_output_features_
    train_rmse = mean_squared_error(y_train, pipe.fit(X_train, y_train).predict(X_train)) ** 0.5
    rows.append({"degree": degree, "features": n_features,
                 "train RMSE": train_rmse, "CV RMSE": cv_rmse(pipe, FEATURES)})
by_degree = pd.DataFrame(rows).set_index("degree")
print(by_degree.round(3))

ax = by_degree[["train RMSE", "CV RMSE"]].plot(marker="o", figsize=(8, 3.5))
ax.set_ylabel("RMSE (MW)")
ax.set_title("More flexibility: training error keeps falling, CV error does not")
plt.show()

**Read the output:** degree 2 cuts the cross-validated RMSE from 4.57 to 4.27 MW, and each degree up to 5 helps a
little more (4.09 MW, with 125 features). After that, the **training** error keeps falling, because the model has
more and more features with which to fit the training hours, while the **cross-validated** error turns sharply
upwards: 4.27 at degree 7 and 4.88 at degree 8 (494 features). The model has started fitting noise in the
training hours that does not repeat in new ones. This is **overfitting**, and the widening gap between the two
lines is how you spot it.

We choose the degree with the lowest CV error (5), not the lowest training error (8). With 7,654 training hours
the model can afford a lot of flexibility before it overfits; with a few hundred rows, it would start much
sooner.

## Step 9: Final check on the test set

Fit the chosen model on the whole training set and score it, once, on the test hours.

In [ ]:
best_degree = by_degree["CV RMSE"].idxmin()
final = make_pipeline(PolynomialFeatures(best_degree, include_bias=False), StandardScaler(), LinearRegression())
final.fit(X_train, y_train)
print(f"chosen degree: {best_degree}")
report("temperature only (Session 1)", y_test, one_input.predict(X_test[["temperature"]]))
report("four inputs, straight", y_test, model.predict(X_test))
report(f"four inputs, degree {best_degree}", y_test, final.predict(X_test))

residuals = y_test - final.predict(X_test)
bands = pd.cut(X_test["temperature"], bins=[0, 10, 15, 20, 25, 30, 40])
print()
print(residuals.groupby(bands, observed=True).agg(["mean", "std"]).round(2))

**Read the output:** the degree-5 model's test RMSE (3.94 MW) is close to its cross-validated estimate (4.09),
which is what honest model selection should give. Step by step, the forecast error has dropped from 5.38 MW (one
input) to 4.50 (four inputs) to 3.94 MW (with curvature), and R² from 0.900 to 0.947. The mean residual by
temperature band is now within ±0.7 MW everywhere, including above 30 °C, where Session 1's line was off by
4.8 MW.

## What this session hands to the next

- **The scikit-learn workflow:** `fit` / `predict`, pipelines that bundle preprocessing with the model,
  cross-validation for every choice, the test set once at the end.
- **How to read coefficients:** "holding the others constant", and only comparable after standardising.
- **A limit of linear models:** to fit a curve we had to *invent* the right extra features. With four inputs that
  is easy; with eight inputs and unknown interactions it is guesswork. Session 3 moves to a harder problem where
  tree-based models find the shape by themselves, and shows how to explain what they learned.

## Common mistakes and fixes

| Mistake | What goes wrong | Fix |
| --- | --- | --- |
| Reading a coefficient as the input's effect on its own | Wrong story when inputs are correlated (humidity's sign flips) | Say "holding the others constant"; compare with the single-input fit |
| Ranking inputs by raw coefficients | Units differ, so sizes are not comparable | Standardise inputs first, or use permutation importance (Session 3) |
| Choosing inputs or degree on the test set | The test score becomes optimistic | Use cross-validation on the training set |
| Scaling before splitting, outside a pipeline | Test-set statistics leak into training | Put the scaler inside `make_pipeline` |
| Forgetting the minus sign on `neg_*` scorers | "Best" model is the worst one | `-cross_val_score(..., scoring="neg_root_mean_squared_error")` |
| Raising the polynomial degree until training error is tiny | Overfitting | Pick the degree with the lowest CV error |

## Exercises

- [ ] Repeat Step 7 with 10 folds instead of 5. Does the ranking of the subsets change?
- [ ] Fit degree 2 on `temperature` alone. How much of the improvement in Step 8 comes from the curve in
  temperature, and how much from interactions between inputs?
- [ ] Replace `LinearRegression` with `Ridge(alpha=10)` (from `sklearn.linear_model`) in the degree-8 pipeline.
  What happens to the CV RMSE, and why?
- [ ] Use `final` to forecast the output for a 35 °C day with vacuum 70, pressure 1010 and humidity 50.

Next, Session 3 predicts the strength of concrete, where the relationships are strongly non-linear: random forests
and gradient boosting beat linear regression by a wide margin, and you learn how to explain their predictions.